In [9]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.special import inv_boxcox
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import haversine_distances
import joblib

In [10]:
class BoxCoxTargetTransformer:
    def __init__(self):
        self.lambda_ = None

    def fit(self, y):
        y = np.array(y)
        _, self.lambda_ = stats.boxcox(y + 1e-6)
        return self

    def transform(self, y):
        return stats.boxcox(np.array(y) + 1e-6, lmbda=self.lambda_)

    def inverse_transform(self, y_bc):
        return inv_boxcox(y_bc, self.lambda_)

def add_distance(X):
    X = X.copy()
    coords = np.radians(X[['latitude', 'longitude']].values)
    center = np.radians([[40.4168, -3.7038]])
    X['distance_to_center_km'] = haversine_distances(coords, center) * 6371
    return X

distance_transformer = FunctionTransformer(add_distance, validate=False)

In [11]:
df = pd.read_csv("df_precios.csv")  # ← asegúrate de que está en la misma carpeta

cols_to_drop = ["id", "name", "host_id", "host_name", "last_review", "license"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df = df.drop_duplicates().dropna()

X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")

✅ Datos cargados: 16923 filas, 13 columnas


In [12]:
y_transformer = BoxCoxTargetTransformer()
y_train_bc = y_transformer.fit(y_train).transform(y_train)
y_test_bc  = y_transformer.transform(y_test)

print("✅ Target transformado con Box-Cox")

✅ Target transformado con Box-Cox


In [13]:
categorical_cols = ['neighbourhood_group', 'neighbourhood', 'room_type']
numerical_cols   = [
    'availability_365', 'calculated_host_listings_count',
    'reviews_per_month', 'number_of_reviews',
    'minimum_nights', 'distance_to_center_km'
]

preprocessor = Pipeline([
    ('add_distance', distance_transformer),
    ('col_transform', ColumnTransformer([
        ('num', Pipeline([
            ('log1p',  FunctionTransformer(np.log1p, validate=False)),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]))
])

pipeline_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor',    LinearRegression())
])

pipeline_lr.fit(X_train, y_train_bc)
print("✅ Modelo entrenado")

✅ Modelo entrenado


In [14]:
y_pred_bc   = pipeline_lr.predict(X_test)
y_pred_real = y_transformer.inverse_transform(y_pred_bc)

mae  = mean_absolute_error(y_test, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_real))  # ← corregido
r2   = r2_score(y_test_bc, y_pred_bc)

print("📊 Linear Regression Pipeline")
print(f"   MAE:  {mae:.2f} €")
print(f"   RMSE: {rmse:.2f} €")
print(f"   R²:   {r2:.4f}")

📊 Linear Regression Pipeline
   MAE:  68.97 €
   RMSE: 392.71 €
   R²:   0.4825


In [15]:
joblib.dump(
    {"pipeline": pipeline_lr, "target_transformer": y_transformer},
    "pipeline_linear_regression_airbnb.pkl"
)
print("✅ pipeline_linear_regression_airbnb.pkl guardado en la misma carpeta")

✅ pipeline_linear_regression_airbnb.pkl guardado en la misma carpeta
